# Differenced GTVP diagnostic

Companion to `GTVP.ipynb` (Peiqin's ARRF/GTVP/LSTM subsection). The committed
levels-based diagnostic battery in `GTVP.ipynb` found severe multicollinearity
between `hprice` and `cc` in log levels (global corr = 0.9869; local
leaf-condition numbers with 100% of leaves > kappa=100, median 1150.7; global
condition number 860.8) and a null predictive result (Clark-West n=165,
CW=-1.61, one-sided p=0.9464 against a leaf-weighted intercept-only nested
baseline).

**This notebook does not modify or re-run `GTVP.ipynb`.** It tests whether that
collinearity is trend-driven — an artefact of `hprice` and `cc` both being
nonstationary/upward-trending in log levels — by re-running the same diagnostic
battery with `hprice` and `cc` first-differenced. `rate` is kept in levels (see
flag below). A global VIF check on `d_hprice`, `d_cc`, `rate` (not shown here)
came back clean (VIF 1.19 / 1.21 / 1.06), motivating this leaf-level check.

## Data preparation

Mirrors `GTVP.ipynb` cells 3–8 exactly (same source file, same log/deflate transform), so `hprice`/`cc` here are the identical log(level/gdp_def) series the committed notebook feeds into its design matrix `X`, before any differencing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
import statsmodels.api as sm
from scipy import stats

In [ ]:
df = pd.read_csv('england_master.csv')

In [ ]:
df['date'] = pd.to_datetime(df['Unnamed: 0'].str.replace('Q', '-Q'))
df.set_index('date', inplace=True)

In [ ]:
df_clean = df[['starts', 'hprice', 'cc', 'rate', 'vol', 'gdp_def']].copy()

In [ ]:
df_clean['starts_lag1'] = df_clean['starts'].shift(1)
df_clean['starts_lag4'] = df_clean['starts'].shift(4)
df_clean['vol_lag1'] = df_clean['vol'].shift(1)
df_clean['rate_lag1'] = df_clean['rate'].shift(1)
df_clean['time_trend'] = np.arange(len(df_clean))

In [ ]:
df_clean = df_clean.dropna()

In [ ]:
df_clean['starts'] = np.log(df_clean['starts'])
df_clean['hprice'] = np.log(df_clean['hprice'] / df_clean['gdp_def'])  # deflated by gdp_def, matching GTVP.ipynb
df_clean['cc'] = np.log(df_clean['cc'] / df_clean['gdp_def'])  # deflated by gdp_def, matching GTVP.ipynb
print(f"df_clean (pre-difference, log-deflated levels): {df_clean.shape}, {df_clean.index.min().date()} to {df_clean.index.max().date()}")

### First-difference `hprice`, `cc`; keep `rate` in levels

`d_hprice`/`d_cc` are the quarter-on-quarter first differences of the exact
log-deflated series above (`Δlog(hprice/gdp_def)`, `Δlog(cc/gdp_def)`) — i.e.
approximately the QoQ growth rate of deflated house prices / construction
costs. Differencing drops the first observation (NaN), so the working sample
here is one quarter shorter than `GTVP.ipynb`'s.

**Flag (per instructions, not guessed):** `rate` is left in levels. No
stationarity check (e.g. ADF/KPSS) on `rate` was run as part of this
diagnostic to confirm that's safe — this mirrors `GTVP.ipynb`'s treatment of
`rate` and is carried over unchanged, but if a future unit-root check finds
`rate` non-stationary over this sample, this choice should be revisited.

In [ ]:
df_diff = df_clean.copy()
df_diff['d_hprice'] = df_diff['hprice'].diff()
df_diff['d_cc'] = df_diff['cc'].diff()
df_diff = df_diff.dropna(subset=['d_hprice', 'd_cc'])
print(f"df_diff (post-difference): {df_diff.shape}, {df_diff.index.min().date()} to {df_diff.index.max().date()}")

In [ ]:
y_diff = df_diff['starts']
X_diff = df_diff[['d_hprice', 'd_cc', 'rate']]
X_diff = sm.add_constant(X_diff)
S_diff = df_diff[['starts_lag1', 'starts_lag4', 'vol_lag1', 'rate_lag1', 'time_trend']]

## The macroeconomic RF

Same RF partition logic and hyperparameters as `GTVP.ipynb` (`n_estimators=500, min_samples_leaf=15, random_state=42`), refit on `S_diff`/`y_diff` because differencing changes the sample by one row versus the levels version. `S` itself is unchanged in construction (starts/vol/rate lags + time trend) — only the local-regression design matrix `X` changes.

In [ ]:
rf_diff = RandomForestRegressor(n_estimators=500, min_samples_leaf=15, random_state=42)
rf_diff.fit(S_diff, y_diff)
leaf_assignments_diff = rf_diff.apply(S_diff)

In [ ]:
valid_idx_diff = df_diff.index
gtvps_diff = pd.DataFrame(index=valid_idx_diff, columns=X_diff.columns, dtype=float)
predicted_y_diff = pd.Series(index=valid_idx_diff, dtype=float)

Ridge lambda selection: identical leaf-weighted CV procedure to `GTVP.ipynb` cell 16 (pre-2010Q1 subsample, same alpha grid, same leaf-co-occurrence weighting), applied to the differenced design matrix.

In [ ]:
X_diff_no_const = X_diff.drop(columns='const')
pre_idx_diff = np.where(df_diff.index < pd.Timestamp('2010-01-01'))[0]
alphas = np.logspace(-3, 5, 17)
tscv = TimeSeriesSplit(n_splits=5)
cv_errors = {a: [] for a in alphas}
for train_pos, test_pos in tscv.split(pre_idx_diff):
    train_idx = pre_idx_diff[train_pos]
    for t in pre_idx_diff[test_pos]:
        w = np.sum(leaf_assignments_diff[train_idx] == leaf_assignments_diff[t], axis=1)
        if w.sum() == 0:
            continue
        w = w / w.sum()
        for a in alphas:
            m = Ridge(alpha=a).fit(X_diff_no_const.iloc[train_idx], y_diff.iloc[train_idx], sample_weight=w)
            pred = m.predict(X_diff_no_const.iloc[[t]])[0]
            cv_errors[a].append((y_diff.iloc[t] - pred) ** 2)
lam_diff = min(alphas, key=lambda a: np.mean(cv_errors[a]))
print(f"Selected ridge lambda (differenced, leaf-weighted CV, pre-2010Q1): {lam_diff}")

In [ ]:
for t_idx in range(len(valid_idx_diff)):
    current_leaves = leaf_assignments_diff[t_idx, :]
    weights = np.sum(leaf_assignments_diff == current_leaves, axis=1)
    weights = weights / weights.sum()
    ridge_model = Ridge(alpha=lam_diff).fit(X_diff_no_const, y_diff, sample_weight=weights)
    gtvps_diff.iloc[t_idx] = np.concatenate([[ridge_model.intercept_], ridge_model.coef_])
    predicted_y_diff.iloc[t_idx] = ridge_model.predict(X_diff_no_const.iloc[[t_idx]])[0]
gtvps_diff = gtvps_diff.astype(float)

## Diagnostic battery (differenced hprice/cc)

Mirrors `GTVP.ipynb` cells 25–40, on `d_hprice`/`d_cc`/`rate` instead of `hprice`/`cc`/`rate`.

### 1. Collinearity diagnostics

In [ ]:
corr_dhprice_dcc = df_diff[['d_hprice', 'd_cc']].corr().loc['d_hprice', 'd_cc']
print(f"Global correlation between d_hprice and d_cc: {corr_dhprice_dcc:.4f}")

In [ ]:
n_trees = leaf_assignments_diff.shape[1]
leaf_conds_diff = []
for t in range(n_trees):
    tree_leaves = leaf_assignments_diff[:, t]
    for leaf_id in np.unique(tree_leaves):
        idx = np.where(tree_leaves == leaf_id)[0]
        if len(idx) < X_diff.shape[1]:
            continue
        Xl = X_diff.iloc[idx].values
        singular_values = np.linalg.svd(Xl, compute_uv=False)
        if singular_values.min() > 0:
            leaf_conds_diff.append(singular_values.max() / singular_values.min())
leaf_conds_diff = np.array(leaf_conds_diff)
pct_above_100_diff = 100 * np.mean(leaf_conds_diff > 100)
print(f"Leaves evaluated (>= {X_diff.shape[1]} obs, across {n_trees} trees): {len(leaf_conds_diff)}")
print(f"Local design-matrix condition number - min: {leaf_conds_diff.min():.1f}, median: {np.median(leaf_conds_diff):.1f}, max: {leaf_conds_diff.max():.1f}")
print(f"% of leaves with condition number > 100: {pct_above_100_diff:.1f}%")

In [ ]:
global_singular_values_diff = np.linalg.svd(X_diff.values, compute_uv=False)
global_cond_diff = global_singular_values_diff.max() / global_singular_values_diff.min()
print(f"Global condition number of design matrix X_diff (const, d_hprice, d_cc, rate): {global_cond_diff:.1f}")

### 2. Sample-size sufficiency check

In [ ]:
ess_per_point_diff = np.empty(len(valid_idx_diff))
for t_idx in range(len(valid_idx_diff)):
    current_leaves = leaf_assignments_diff[t_idx, :]
    weights = np.sum(leaf_assignments_diff == current_leaves, axis=1).astype(float)
    weights = weights / weights.sum()
    ess_per_point_diff[t_idx] = 1.0 / np.sum(weights ** 2)

n_params = 4  # const, d_hprice, d_cc, rate
pct_below_params_diff = 100 * np.mean(ess_per_point_diff < n_params)
pct_below_4x_diff = 100 * np.mean(ess_per_point_diff < 4 * n_params)
print(f"Effective sample size (ESS) per local regression - min: {ess_per_point_diff.min():.1f}, median: {np.median(ess_per_point_diff):.1f}, max: {ess_per_point_diff.max():.1f}")
print(f"% of local regressions with ESS < {n_params} (params): {pct_below_params_diff:.1f}%")
print(f"% of local regressions with ESS < {4 * n_params} (4x params): {pct_below_4x_diff:.1f}%")

### 3. Predictive performance comparison

Same nested restriction as `GTVP.ipynb`: a leaf-weighted intercept-only baseline using the same leaf-co-occurrence weights as the differenced GTVP fit.

In [ ]:
tscv_eval = TimeSeriesSplit(n_splits=5)
n_obs_diff = len(y_diff)
cv_records_diff = []
for fold_id, (train_pos, test_pos) in enumerate(tscv_eval.split(np.arange(n_obs_diff))):
    y_train = y_diff.iloc[train_pos]
    for t in test_pos:
        w = np.sum(leaf_assignments_diff[train_pos] == leaf_assignments_diff[t], axis=1).astype(float)
        if w.sum() == 0:
            w = np.ones(len(train_pos))
        w = w / w.sum()
        restricted_pred = np.average(y_train, weights=w)
        m = Ridge(alpha=lam_diff).fit(X_diff_no_const.iloc[train_pos], y_train, sample_weight=w)
        gtvp_pred = m.predict(X_diff_no_const.iloc[[t]])[0]
        cv_records_diff.append({
            'fold': fold_id,
            'y_true': y_diff.iloc[t],
            'gtvp_pred': gtvp_pred,
            'restricted_pred': restricted_pred,
        })
cv_df_diff = pd.DataFrame(cv_records_diff)
cv_df_diff['gtvp_sq_err'] = (cv_df_diff['y_true'] - cv_df_diff['gtvp_pred']) ** 2
cv_df_diff['restricted_sq_err'] = (cv_df_diff['y_true'] - cv_df_diff['restricted_pred']) ** 2
print(f"Out-of-sample folds: {cv_df_diff['fold'].nunique()}, total held-out observations: {len(cv_df_diff)}")

In [ ]:
gtvp_cv_mse_diff = cv_df_diff['gtvp_sq_err'].mean()
restricted_cv_mse_diff = cv_df_diff['restricted_sq_err'].mean()
print(f"CV MSE - ridge-penalized GTVP, differenced (alpha={lam_diff}): {gtvp_cv_mse_diff:.6f}")
print(f"CV MSE - leaf-weighted intercept-only baseline (nested): {restricted_cv_mse_diff:.6f}")
print(f"Difference (GTVP - intercept-only): {gtvp_cv_mse_diff - restricted_cv_mse_diff:.6f}")

t_stat_diff, p_val_ttest_diff = stats.ttest_rel(cv_df_diff['gtvp_sq_err'], cv_df_diff['restricted_sq_err'])
print(f"Paired t-test on per-observation squared-error differences: t={t_stat_diff:.4f}, two-sided p={p_val_ttest_diff:.4f}")

### 4. Formal significance test (Clark-West)

In [ ]:
cw_terms_diff = cv_df_diff['restricted_sq_err'] - cv_df_diff['gtvp_sq_err'] + (cv_df_diff['restricted_pred'] - cv_df_diff['gtvp_pred']) ** 2
n_cw_diff = len(cw_terms_diff)
cw_mean_diff = cw_terms_diff.mean()
cw_se_diff = cw_terms_diff.std(ddof=1) / np.sqrt(n_cw_diff)
cw_stat_diff = cw_mean_diff / cw_se_diff
p_value_cw_diff = 1 - stats.norm.cdf(cw_stat_diff)
print("Clark-West test (differenced GTVP vs intercept-only restricted model):")
print(f"  n = {n_cw_diff}")
print(f"  CW statistic = {cw_stat_diff:.4f}")
print(f"  one-sided p-value (H1: GTVP improves on restricted model) = {p_value_cw_diff:.4f}")
if p_value_cw_diff < 0.05:
    print("Reject H0 at the 5% level: differenced GTVP shows a statistically significant improvement over the intercept-only model.")
else:
    print("Fail to reject H0 at the 5% level: no statistically significant evidence that differenced GTVP improves on the intercept-only model.")

### 5. Coefficient magnitude summary

In [ ]:
coef_summary_diff = gtvps_diff[['d_hprice', 'd_cc', 'rate']].abs().agg(['median', 'mean', 'std', 'max'])
print("Magnitude of differenced-GTVP slope coefficients (absolute value, across all quarters):")
print(coef_summary_diff)
print()
print(f"Selected ridge penalty lambda (differenced) = {lam_diff:.1f}, at the top edge of the search grid ({alphas.min():.1e} to {alphas.max():.1e}) -- same edge-of-grid behaviour as the levels version.")
print(f"Global correlation(d_hprice, d_cc) = {corr_dhprice_dcc:.4f}; median local design-matrix condition number = {np.median(leaf_conds_diff):.1f} ({pct_above_100_diff:.1f}% of leaves > 100).")

## Levels vs. differenced — side-by-side summary

Levels-version numbers below are copied from the committed `GTVP.ipynb`
diagnostic battery (commit `10a19a1`) for reference — `GTVP.ipynb` itself was
not re-run to produce them.

In [ ]:
summary = pd.DataFrame({
    'levels (GTVP.ipynb, committed)': {
        'global corr(hprice/cc or d_hprice/d_cc)': 0.9869,
        'local leaf cond# median': 1150.7,
        'local leaf cond# % > 100': 100.0,
        'global design-matrix cond#': 860.8,
        'ridge lambda selected': 100000.0,
        'Clark-West n': 165,
        'Clark-West stat': -1.61,
        'Clark-West one-sided p': 0.9464,
    },
    'differenced (this notebook)': {
        'global corr(hprice/cc or d_hprice/d_cc)': round(corr_dhprice_dcc, 4),
        'local leaf cond# median': round(float(np.median(leaf_conds_diff)), 1),
        'local leaf cond# % > 100': round(pct_above_100_diff, 1),
        'global design-matrix cond#': round(global_cond_diff, 1),
        'ridge lambda selected': lam_diff,
        'Clark-West n': n_cw_diff,
        'Clark-West stat': round(cw_stat_diff, 2),
        'Clark-West one-sided p': round(p_value_cw_diff, 4),
    },
})
print(summary)
print()
print("Reading: differencing collapses the GLOBAL correlation (0.99 -> ~0.4) and roughly")
print("halves the local leaf-level condition numbers, consistent with part of the levels-")
print("version collinearity being trend-driven. But local leaves remain ill-conditioned")
print("(100% still > kappa=100; global cond# still ~4x a conventional kappa=30 rule of")
print("thumb), the ridge penalty still saturates the top of the search grid, and the")
print("Clark-West predictive test is essentially unchanged (p~0.94 either way). So the")
print("trend-driven-artefact hypothesis is only PARTIALLY supported: differencing removes")
print("the trend component of the collinearity but a residual short-run comovement between")
print("QoQ house-price and construction-cost growth remains, and it is enough on its own to")
print("keep local leaf regressions ill-conditioned and GTVP's predictive value null.")